# ID3 Decision Tree Algorithm

This notebook demonstrates the **ID3 (Iterative Dichotomiser 3)** algorithm for building a decision tree, implemented from scratch using **Entropy** and **Information Gain**.

**Dataset:** The classic *Play Tennis* dataset — 14 records describing weather conditions (`Outlook`, `Temperature`, `Humidity`, `Wind`) and whether a game of tennis was played (`PlayTennis`).

**Steps covered:**
1. Load the dataset
2. Compute Entropy and Information Gain
3. Build the decision tree recursively (ID3)
4. Visualize the learned tree
5. Classify a new, unseen sample

## 1. Load the Dataset

In [1]:
import pandas as pd
import numpy as np
from pprint import pprint

# The classic "Play Tennis" dataset
data = {
    'Outlook':    ['Sunny','Sunny','Overcast','Rain','Rain','Rain','Overcast',
                    'Sunny','Sunny','Rain','Sunny','Overcast','Overcast','Rain'],
    'Temperature':['Hot','Hot','Hot','Mild','Cool','Cool','Cool',
                    'Mild','Cool','Mild','Mild','Mild','Hot','Mild'],
    'Humidity':   ['High','High','High','High','Normal','Normal','Normal',
                    'High','Normal','Normal','Normal','High','Normal','High'],
    'Wind':       ['Weak','Strong','Weak','Weak','Weak','Strong','Strong',
                    'Weak','Weak','Weak','Strong','Strong','Weak','Strong'],
    'PlayTennis': ['No','No','Yes','Yes','Yes','No','Yes',
                    'No','Yes','Yes','Yes','Yes','Yes','No']
}
df = pd.DataFrame(data)
target_attribute = 'PlayTennis'

print("Training Data:")
print(df)

Training Data:
     Outlook Temperature Humidity    Wind PlayTennis
0      Sunny         Hot     High    Weak         No
1      Sunny         Hot     High  Strong         No
2   Overcast         Hot     High    Weak        Yes
3       Rain        Mild     High    Weak        Yes
4       Rain        Cool   Normal    Weak        Yes
5       Rain        Cool   Normal  Strong         No
6   Overcast        Cool   Normal  Strong        Yes
7      Sunny        Mild     High    Weak         No
8      Sunny        Cool   Normal    Weak        Yes
9       Rain        Mild   Normal    Weak        Yes
10     Sunny        Mild   Normal  Strong        Yes
11  Overcast        Mild     High  Strong        Yes
12  Overcast         Hot   Normal    Weak        Yes
13      Rain        Mild     High  Strong         No


## 2. Entropy and Information Gain

**Entropy** measures the impurity/uncertainty of a set of labels:

$$Entropy(S) = -\sum_{i} p_i \log_2(p_i)$$

**Information Gain** measures the reduction in entropy achieved by splitting on an attribute:

$$Gain(S, A) = Entropy(S) - \sum_{v \in Values(A)} \frac{|S_v|}{|S|} Entropy(S_v)$$

ID3 picks, at each step, the attribute that **maximizes Information Gain**.

In [2]:
def entropy(labels):
    values, counts = np.unique(labels, return_counts=True)
    probs = counts / counts.sum()
    return -np.sum(probs * np.log2(probs))


def info_gain(df, attribute, target_attribute):
    total_entropy = entropy(df[target_attribute])
    values, counts = np.unique(df[attribute], return_counts=True)
    weighted_entropy = 0
    for v, c in zip(values, counts):
        subset = df[df[attribute] == v]
        weighted_entropy += (c / len(df)) * entropy(subset[target_attribute])
    return total_entropy - weighted_entropy

In [3]:
features = [col for col in df.columns if col != target_attribute]

print("Information Gain of each attribute (at root):")
for f in features:
    print(f"  {f}: {info_gain(df, f, target_attribute):.4f}")

Information Gain of each attribute (at root):
  Outlook: 0.2467
  Temperature: 0.0292
  Humidity: 0.1518
  Wind: 0.0481


`Outlook` has the highest Information Gain (0.2467), so it becomes the **root** of the tree. ID3 then recurses on each branch, choosing the best remaining attribute each time.

## 3. The ID3 Algorithm (Recursive Tree Building)

In [4]:
def id3(df, original_df, features, target_attribute, parent_class=None):
    # 1. If all target values are the same, return that value (pure leaf)
    if len(np.unique(df[target_attribute])) <= 1:
        return np.unique(df[target_attribute])[0]

    # 2. If dataset is empty, return the mode target value of the original data
    elif len(df) == 0:
        return np.unique(original_df[target_attribute])[
            np.argmax(np.unique(original_df[target_attribute], return_counts=True)[1])
        ]

    # 3. If no more features to split on, return the parent node's majority class
    elif len(features) == 0:
        return parent_class

    # 4. Otherwise grow the tree
    else:
        parent_class = np.unique(df[target_attribute])[
            np.argmax(np.unique(df[target_attribute], return_counts=True)[1])
        ]

        # Choose the attribute with the highest information gain
        gains = [info_gain(df, feature, target_attribute) for feature in features]
        best_feature_index = np.argmax(gains)
        best_feature = features[best_feature_index]

        tree = {best_feature: {}}
        remaining_features = [f for f in features if f != best_feature]

        for value in np.unique(df[best_feature]):
            sub_data = df[df[best_feature] == value]
            subtree = id3(sub_data, original_df, remaining_features,
                          target_attribute, parent_class)
            tree[best_feature][value] = subtree

        return tree

## 4. Build and Visualize the Tree

In [5]:
tree = id3(df, df, features, target_attribute)

print("Learned Decision Tree (nested dict form):")
pprint(tree)

Learned Decision Tree (nested dict form):
{'Outlook': {'Overcast': 'Yes',
             'Rain': {'Wind': {'Strong': 'No', 'Weak': 'Yes'}},
             'Sunny': {'Humidity': {'High': 'No', 'Normal': 'Yes'}}}}


In [6]:
def print_tree(tree, indent=""):
    """Pretty-print the decision tree in an indented, human-readable form."""
    if not isinstance(tree, dict):
        print(indent + "-> " + str(tree))
        return
    attribute = next(iter(tree))
    for value, subtree in tree[attribute].items():
        print(f"{indent}[{attribute} = {value}]")
        print_tree(subtree, indent + "    ")

print_tree(tree)

[Outlook = Overcast]
    -> Yes
[Outlook = Rain]
    [Wind = Strong]
        -> No
    [Wind = Weak]
        -> Yes
[Outlook = Sunny]
    [Humidity = High]
        -> No
    [Humidity = Normal]
        -> Yes


**Interpretation of the tree:**
- If `Outlook = Overcast` → always **Play Tennis: Yes**
- If `Outlook = Rain` → decision depends on `Wind` (Strong → No, Weak → Yes)
- If `Outlook = Sunny` → decision depends on `Humidity` (High → No, Normal → Yes)
- Notice `Temperature` was never used — it added the least information.

## 5. Classify a New Sample

In [7]:
def classify(sample, tree):
    if not isinstance(tree, dict):
        return tree
    attribute = next(iter(tree))
    value = sample.get(attribute)
    subtree = tree[attribute].get(value, None)
    if subtree is None:
        return "Unknown (value not seen during training)"
    return classify(sample, subtree)


new_sample = {
    'Outlook': 'Sunny',
    'Temperature': 'Cool',
    'Humidity': 'High',
    'Wind': 'Strong'
}

prediction = classify(new_sample, tree)
print("New Sample:", new_sample)
print("Predicted class (PlayTennis):", prediction)

New Sample: {'Outlook': 'Sunny', 'Temperature': 'Cool', 'Humidity': 'High', 'Wind': 'Strong'}
Predicted class (PlayTennis): No


**Trace of the classification:**
1. `Outlook = Sunny` → go to the `Humidity` sub-tree
2. `Humidity = High` → leaf node = **No**

So the model predicts the person will **not** play tennis, even though `Wind = Strong` — because once `Outlook = Sunny` is decided, only `Humidity` matters (that branch never even looks at `Wind`).

### Try it yourself
Change the values in `new_sample` above and re-run the last cell to classify different conditions.